In [ ]:
#| default_exp machine_learning.tokenize.def_and_notat_token_classification

Previous, `trouver` just had functionalities for using ML models to identify newly introduced notations in text and for gathering data to train such models. Moreover, such models were merely classification models, and using these models to identify newly introduced notations had a lot of computational redundancies.

This module aims to provide the same functionalities for both definitions and notations by training and using token classification models instead.

In [ ]:
# TODO: Create a new module dedicated to definition and notation identification and move approparite functions over there. 

In [ ]:
#| export
from collections.abc import Callable
import copy
from itertools import pairwise
import os 
from os import PathLike
from pathlib import Path
import random
from typing import Literal, Optional, TypedDict, Union
import warnings

import bs4
from openai import OpenAI
from pydantic import BaseModel
import regex
from transformers import BatchEncoding, pipelines, PreTrainedTokenizer, PreTrainedTokenizerFast

from trouver.helper import is_not_space_and_not_punc, split_string_at_indices
from trouver.helper.definition_and_notation import double_asterisk_indices, notation_asterisk_indices
from trouver.helper.html import (
    add_HTML_tag_data_to_raw_text, add_space_to_lt_symbols_without_space, remove_html_tags_in_text, StrAndHTMLTagsWithIndices,
    HTMLTagWithIndices)
from trouver.helper.latex.core import _is_balanced_braces, _first_curly_bracket, _last_curly_bracket
from trouver.helper.latex.augment import (
    augment_text, change_font_styles_at_random, change_greek_letters_at_random, dollar_sign_manipulation, push_dollar_sign, random_char_modification, random_latex_command_removal, random_word_removal, remove_font_styles_at_random, remove_math_keywords)

from trouver.helper.regex import latex_indices, replace_string_by_indices
from trouver.obsidian.file import MarkdownFile, MarkdownLineEnum
from trouver.personal_vault.note_processing import process_standard_information_note
from trouver.obsidian.vault import VaultNote



In [ ]:
from unittest import mock
import shutil
import tempfile

from datasets import ClassLabel, Dataset, Features, Sequence, Value
from transformers import AutoTokenizer
from fastcore.test import *

from trouver.helper.tests import _test_directory

## Parse data from information notes

In [ ]:
#| export
def convert_double_asterisks_to_html_tags(
        text: str
        ) -> str:
    """
    Replace the double asterisks, which signify definitions and notations,
    in `text` with HTML tags.
    """
    double_asts = double_asterisk_indices(text)
    replacement_html_tags = [
        _html_tag_from_double_ast(text[start:end])
        for start, end in double_asts]
    return replace_string_by_indices(
        text, double_asts, replacement_html_tags)


def _html_tag_from_double_ast(
        double_ast_string: str # Starts and ends with double asts
        ) -> str:
    """
    Get the HTML tag representing definition or notation data from
    a string surrounded by double asterisks.

    This is used in the `_convert_double_asterisks_to_html_tags` function.
    """
    no_asts = double_ast_string[2:-2]
    if notation_asterisk_indices(double_ast_string):
        return f'<span notation="">{no_asts}</span>'
    else:
        return f'<b definition="">{no_asts}</b>'

In [ ]:
print(convert_double_asterisks_to_html_tags("**hi**. Here is a notation **$asdf$**"))
test_eq(convert_double_asterisks_to_html_tags("**hi**. Here is a notation **$asdf$**"), '<b definition="">hi</b>. Here is a notation <span notation="">$asdf$</span>')

<b definition="">hi</b>. Here is a notation <span notation="">$asdf$</span>


In [ ]:
print(convert_double_asterisks_to_html_tags("**$M^*$**."))

<span notation="">$M^*$</span>.


In [ ]:
#| export
def raw_text_with_html_tags_from_markdownfile(
        mf: MarkdownFile,
        vault: PathLike
        ) -> str:
    """
    Process the `MarkdownFile`, replacing the double asterisk surrounded
    text indicating definitions and notations to be HTML tags instead.
    """
    mf = process_standard_information_note(
        mf, vault, remove_double_asterisks=False,
        remove_html_tags=False)
    return convert_double_asterisks_to_html_tags(str(mf))


In [ ]:
#| hide

# TODO: 
# I want to make sure that footnotes are getting properly removed.
mf = MarkdownFile.from_string(
    r"""---
aliases: []
tags: []
---
# Something  

Some kind of potato[^2]

[^2]: Some footnote

[[link_to_note|Some link]]


# See Also
# Meta
## References and Citations
""") 
raw_text_with_html_tags_from_markdownfile(mf, None)

'Some kind of potato[^2]\n\n[^2]: Some footnote\n\nSome link\n'

In [ ]:
#| hide
mf = MarkdownFile.from_string(
    r"""---
aliases: []
tags: []
---
# Galois group of a separable and normal finite field extension

Let $L/K$ be a separable and normal finite field extension. Its <b definition="Galois group of a separable and normal finite field extension">Galois group</b> <span notation="">$\operatorname{Gal}(L/K)$</span> is...

# Galois group of a separable and normal profinite field extension

In fact, the notion of a Galois group can be defined for profinite field extensions. Given a separable and normal profinite field extension $L/K$, say that
$L = \varinjlim_i L_i$ where $L_i/K$ are finite extensions. Its **Galois group** **$\operatorname{Gal}(L/K)$**

# See Also
# Meta
## References and Citations
""")

In the following example, let `mf` be the following `MarkdownFile`:

In [ ]:
print(str(mf))

---
aliases: []
tags: []
---
# Galois group of a separable and normal finite field extension

Let $L/K$ be a separable and normal finite field extension. Its <b definition="Galois group of a separable and normal finite field extension">Galois group</b> <span notation="">$\operatorname{Gal}(L/K)$</span> is...

# Galois group of a separable and normal profinite field extension

In fact, the notion of a Galois group can be defined for profinite field extensions. Given a separable and normal profinite field extension $L/K$, say that
$L = \varinjlim_i L_i$ where $L_i/K$ are finite extensions. Its **Galois group** **$\operatorname{Gal}(L/K)$**

# See Also
# Meta
## References and Citations


The `raw_text_with_html_tags_from_markdownfile` function processes the `MarkdownFile` much in the same way as the `process_standard_information_note` function, except it 1. preserves HTML tags, and 2. replaces text surrounded by double asterisks `**` with HTML tags signifiying whether the text displays a definition or a notation.

In the below example, note that the `vault` parameter is set to `None`; this is fine for this example becaues the `process_standard_information_note` function only needs a `vault` argument when embedded links need to be replaced with text (via the `MarkdownFile.replace_embedded_links_with_text` function), but `mf` has no embedded links.

In [ ]:
print(raw_text_with_html_tags_from_markdownfile(mf, None))

Let $L/K$ be a separable and normal finite field extension. Its <b definition="Galois group of a separable and normal finite field extension">Galois group</b> <span notation="">$\operatorname{Gal}(L/K)$</span> is...

In fact, the notion of a Galois group can be defined for profinite field extensions. Given a separable and normal profinite field extension $L/K$, say that
$L = \varinjlim_i L_i$ where $L_i/K$ are finite extensions. Its <b definition="">Galois group</b> <span notation="">$\operatorname{Gal}(L/K)$</span>



In [ ]:
#| hide
assert '**' not in raw_text_with_html_tags_from_markdownfile(mf, None)

In [ ]:
#| export
class HTMLData(TypedDict):
    note_name: str
    raw_text: str
    tags: list[HTMLTagWithIndices]
    # list[bs4.element.Tag]

In [ ]:
#| export
def html_data_from_note(
        note_or_mf: Union[VaultNote, MarkdownFile], # Either a `VaultNote`` object to a note or a `MarkdownFile` object from which to extra html data.
        vault: Optional[PathLike] = None, # If vault to use when processing the `MarkdownFile` objects (if `note_of_mf` is a `VaultNote`, then this `MarkdownFile` object is created from the text of the note), cf. the `process_standard_information_note` function.
        note_name: Optional[str] = None, # If `note_or_mf` is a `MarkdownFile`, `note_name` should be the name of the note from which the `MarkdownFile` comes from if applicable. If `note_or_mf` is a `VaultNote` object, then `note_name` is ignored and `note_or_mf.name` is used instead.
        ) -> Union[HTMLData, None]: # The keys to the dict are "note_name", "raw_text", "tags". However, `None` is returned if `note` does not exist or the note is marked with auto-generated, unverified data.
    # TODO: implement obtaining multiple datapoints from a single note
    # Via typos for example.
    # TODO: implement various data augmentation techniques
    """Obtain html data for token classification from the information note.

    Currently, the token types mainly revolve around definitions and
    notations.

    If `note` has the tag `_auto/def_and_notat_identified`, then the data
    in the note is assumed to be auto-generated and not verified and
    `None` is returned.

    **Returns**
    - Union[dict, None]
        - The keys-value pairs are 
            - `"note_name"` - The name of the note
            - `"raw_text"` - The raw text to include in the data.
            - `"tags"` - The list with HTML tags carrying definition/notation
              data and their locations in the Raw text. See the second output to
              the function `remove_html_tags_in_text`.
                - Each element of the list is a tuple consisting of a ``bs4.element.Tag``
                  and two ints.
    """
    if isinstance(note_or_mf, VaultNote) and not note_or_mf.exists():
        return None
    if isinstance(note_or_mf, VaultNote):
        mf = MarkdownFile.from_vault_note(note_or_mf)
        note_name = note_or_mf.name
        if vault is None:
            vault = note_or_mf.vault
    else: # isinstance(note_or_mf, MarkdownFile):
        mf = note_or_mf.copy(deep=False)
    if mf.has_tag('_auto/def_and_notat_identified'):
        return None
    raw_text_with_tags = raw_text_with_html_tags_from_markdownfile(mf, vault)
    raw_text, tags_and_locations = remove_html_tags_in_text(raw_text_with_tags)

    return HTMLData(note_name=note_name, raw_text=raw_text, tags=tags_and_locations)

In the following example, we mock a `VaultNote` whose content is that of `mf` in the example for the `raw_text_with_html_tags_from_markdownfile` function. Note that there is some text surrounded by double within `mf` surrounded by double asterisks `**` and some text surrounded by HTML tags to indicate definitions and notations introduced.

In [ ]:
mf = MarkdownFile.from_string(
    r"""---
aliases: []
tags: []
---
# Galois group of a separable and normal finite field extension

Let $L/K$ be a separable and normal finite field extension. Its <b definition="Galois group of a separable and normal finite field extension">Galois group</b> <span notation="">$\operatorname{Gal}(L/K)$</span> is...

# Galois group of a separable and normal profinite field extension

In fact, the notion of a Galois group can be defined for profinite field extensions. Given a separable and normal profinite field extension $L/K$, say that
$L = \varinjlim_i L_i$ where $L_i/K$ are finite extensions. Its **Galois group** **$\operatorname{Gal}(L/K)$**

# See Also
# Meta
## References and Citations
""")

with (mock.patch('__main__.VaultNote') as mock_VaultNote,
      mock.patch('__main__.MarkdownFile.from_vault_note') as mock_from_vault_note,
      mock.patch('__main__.isinstance') as mock_isinstance):
    mock_VaultNote.exists.return_value = True
    mock_VaultNote.name = "Note's name"
    mock_from_vault_note.return_value = mf
    mock_isinstance.return_value = True

    print(f"The following is the text from mf:\n\n{str(mf)}")

    html_data = html_data_from_note(mock_VaultNote, None)
    print(html_data)

    test_eq(html_data['note_name'], "Note's name")
    assert '**' not in html_data['raw_text']
    assert '<' not in html_data['raw_text']  # Test the lack of HTML tags in the raw text

    print(html_data['tags'])
    test_eq(len(html_data['tags']), 4)
    assert isinstance(html_data['tags'][0][0], bs4.element.Tag)
    assert html_data['tags'][0][0].has_attr('definition')
    assert not html_data['tags'][0][0].has_attr('notation')
    assert html_data['tags'][1][0].has_attr('notation')
    assert not html_data['tags'][1][0].has_attr('definition')
    assert html_data['tags'][2][0].has_attr('definition')
    assert not html_data['tags'][2][0].has_attr('notation')
    assert html_data['tags'][3][0].has_attr('notation')
    assert not html_data['tags'][3][0].has_attr('definition')

The following is the text from mf:

---
aliases: []
tags: []
---
# Galois group of a separable and normal finite field extension

Let $L/K$ be a separable and normal finite field extension. Its <b definition="Galois group of a separable and normal finite field extension">Galois group</b> <span notation="">$\operatorname{Gal}(L/K)$</span> is...

# Galois group of a separable and normal profinite field extension

In fact, the notion of a Galois group can be defined for profinite field extensions. Given a separable and normal profinite field extension $L/K$, say that
$L = \varinjlim_i L_i$ where $L_i/K$ are finite extensions. Its **Galois group** **$\operatorname{Gal}(L/K)$**

# See Also
# Meta
## References and Citations
{'note_name': "Note's name", 'raw_text': 'Let $L/K$ be a separable and normal finite field extension. Its Galois group $\\operatorname{Gal}(L/K)$ is...\n\nIn fact, the notion of a Galois group can be defined for profinite field extensions. Given a separable and normal pr

We can also just pass a `MarkdonwFile` object instead of a `VaultNote` object. In this case, we can specify the `note_name` parameter to indicate which note the `MarkdownFile` object came from, if applicable.

In [ ]:
html_data = html_data_from_note(mf, vault=None, note_name="Note's name")
print(html_data)

test_eq(html_data['note_name'], "Note's name")
assert '**' not in html_data['raw_text']
assert '<' not in html_data['raw_text']  # Test the lack of HTML tags in the raw text

print(html_data['tags'])
test_eq(len(html_data['tags']), 4)
assert isinstance(html_data['tags'][0][0], bs4.element.Tag)
assert html_data['tags'][0][0].has_attr('definition')
assert not html_data['tags'][0][0].has_attr('notation')
assert html_data['tags'][1][0].has_attr('notation')
assert not html_data['tags'][1][0].has_attr('definition')
assert html_data['tags'][2][0].has_attr('definition')
assert not html_data['tags'][2][0].has_attr('notation')
assert html_data['tags'][3][0].has_attr('notation')
assert not html_data['tags'][3][0].has_attr('definition')

{'note_name': "Note's name", 'raw_text': 'Let $L/K$ be a separable and normal finite field extension. Its Galois group $\\operatorname{Gal}(L/K)$ is...\n\nIn fact, the notion of a Galois group can be defined for profinite field extensions. Given a separable and normal profinite field extension $L/K$, say that\n$L = \\varinjlim_i L_i$ where $L_i/K$ are finite extensions. Its Galois group $\\operatorname{Gal}(L/K)$\n', 'tags': [HTMLTagWithIndices(tag=<b definition="Galois group of a separable and normal finite field extension">Galois group</b>, start=64, end=76), HTMLTagWithIndices(tag=<span notation="">$\operatorname{Gal}(L/K)$</span>, start=77, end=102), HTMLTagWithIndices(tag=<b definition="">Galois group</b>, start=330, end=342), HTMLTagWithIndices(tag=<span notation="">$\operatorname{Gal}(L/K)$</span>, start=343, end=368)]}
[HTMLTagWithIndices(tag=<b definition="Galois group of a separable and normal finite field extension">Galois group</b>, start=64, end=76), HTMLTagWithIndices(tag

If we do not specify `note_name`, then `None` is used for the `'Note name'` key in the output:

In [ ]:
html_data = html_data_from_note(mf, vault=None, note_name=None)
print(html_data)

assert html_data['note_name'] is None

{'note_name': None, 'raw_text': 'Let $L/K$ be a separable and normal finite field extension. Its Galois group $\\operatorname{Gal}(L/K)$ is...\n\nIn fact, the notion of a Galois group can be defined for profinite field extensions. Given a separable and normal profinite field extension $L/K$, say that\n$L = \\varinjlim_i L_i$ where $L_i/K$ are finite extensions. Its Galois group $\\operatorname{Gal}(L/K)$\n', 'tags': [HTMLTagWithIndices(tag=<b definition="Galois group of a separable and normal finite field extension">Galois group</b>, start=64, end=76), HTMLTagWithIndices(tag=<span notation="">$\operatorname{Gal}(L/K)$</span>, start=77, end=102), HTMLTagWithIndices(tag=<b definition="">Galois group</b>, start=330, end=342), HTMLTagWithIndices(tag=<span notation="">$\operatorname{Gal}(L/K)$</span>, start=343, end=368)]}


For the following example, the note has an HTML tag already with extra data (attributes other than `'definition'` or `'notation'`). We assert that the extra data is preserved. 

In [ ]:
with (mock.patch('__main__.VaultNote') as mock_VaultNote,
      mock.patch('__main__.MarkdownFile.from_vault_note') as mock_from_vault_note,
      mock.patch('__main__.isinstance') as mock_isinstance):
    mock_VaultNote.exists.return_value = True
    mock_VaultNote.name = "Note's name"
    mock_isinstance.return_value = True

    text = r'Let $X$ be a topological space and let $U \subseteq X$ be an subspace. The <b definition="Closure of a subspace of a topological space" typo="dosure of $U$">closure of $U$</b> is defined as...'
    mf = MarkdownFile.from_string(text)
    mock_from_vault_note.return_value = mf
    print(f"The following is the text of the mocked note: \n\n {text}\n\n")

    html_data = html_data_from_note(mock_VaultNote, None)
    print(html_data)
    assert html_data['tags'][0][0].has_attr('typo')
    test_eq(html_data['tags'][0][0].attrs['typo'], 'dosure of $U$')

The following is the text of the mocked note: 

 Let $X$ be a topological space and let $U \subseteq X$ be an subspace. The <b definition="Closure of a subspace of a topological space" typo="dosure of $U$">closure of $U$</b> is defined as...


{'note_name': "Note's name", 'raw_text': 'Let $X$ be a topological space and let $U \\subseteq X$ be an subspace. The closure of $U$ is defined as...', 'tags': [HTMLTagWithIndices(tag=<b definition="Closure of a subspace of a topological space" typo="dosure of $U$">closure of $U$</b>, start=75, end=89)]}


In the following example, the (mocked) note has the `#_auto/def_and_notats_identified` tag to indicate that its definition and notation markings were auto-generated by a model (trained with data processed by the `tokenize_html_data` function) using the `auto_mark_def_and_notats` function. In this case, the `html_data_from_note` function returns `None` to prevent gathering data that is unverified and auto-generated by a model.

In [ ]:
# with (mock.patch('__main__.VaultNote') as mock_VaultNote,
#       mock.patch('__main__.MarkdownFile.from_vault_note') as mock_from_vault_note):
#     mock_VaultNote.exists.return_value = True
#     mock_VaultNote.name = "Note's name"
text = r'''---
tags: [_auto/def_and_notat_identified]
---
Let $X$ be a topological space and let $U \subseteq X$ be an subspace. The <b definition="Closure of a subspace of a topological space" typo="dosure of $U$">closure of $U$</b> is defined as...'''

mf = MarkdownFile.from_string(text)
mock_from_vault_note.return_value = mf
print(f"The following is the text of the mocked note: \n\n{text}\n\n")

html_data = html_data_from_note(note_or_mf=mf)
assert(html_data is None)

The following is the text of the mocked note: 

---
tags: [_auto/def_and_notat_identified]
---
Let $X$ be a topological space and let $U \subseteq X$ be an subspace. The <b definition="Closure of a subspace of a topological space" typo="dosure of $U$">closure of $U$</b> is defined as...




In [ ]:
#| export
def tokenize_html_data(
        html_locus: HTMLData, # An output of `html_data_from_note`
        tokenizer: Union[PreTrainedTokenizer, PreTrainedTokenizerFast],
        max_length: int, # Max length for each sequence of tokens
        ner_tag_from_html_tag: Callable[[bs4.element.Tag], Union[str, 'None']], # takes in a bs4.element.Tag and outputs the ner_tag (as a string or `None`)
        label2id: dict[str, int], # The keys are ner_tag's of the form f"I-{output}" or f"B-{output}" where `output` is an output of `ner_tag_from_html_tag`.
        default_label: str = "O", # The default label for the NER tagging.
        ) -> tuple[list[list[str]], list[list[int]]]: # The first list consists of the tokens and the second list consists of the named entity recognition tags.
    """Actually tokenize the html data outputted by `html_data_from_note`.

    To account for the possibility that the raw text is long,
    this function uses the `tokenizer.batch_encode_plus` function
    to tokenize the text into sequences. 
    """
    tokenized = tokenizer.batch_encode_plus(
        [html_locus["raw_text"]], max_length=max_length, return_overflowing_tokens=True,
        return_offsets_mapping=True, truncation=True)

    default_id = label2id[default_label]        
    ner_ids = [[default_id for _ in seq_input_ids]
               for seq_input_ids in tokenized['input_ids']]
    for tag, start, end in html_locus['tags']:
        ner_tag = ner_tag_from_html_tag(tag)
        if ner_tag is None:
            continue  # `ner_tag` is not of relevant data.
        tuppy = _start_end_seqs_indices_for_html_tag(tokenized, start, end - 1)
        (start_seq, start_index_in_seq), (end_seq, end_index_in_seq) = tuppy
        _set_ner_ids_for_tag(
            ner_ids, start_seq, start_index_in_seq, end_seq, end_index_in_seq,
            label2id, ner_tag)
    # return tokenized["input_ids"], ner_ids
    tokens = [tokenizer.convert_ids_to_tokens(tokens_for_seq)
              for tokens_for_seq in tokenized["input_ids"]]
    return tokens, ner_ids


def _start_end_seqs_indices_for_html_tag(
        tokenized: BatchEncoding,
        tag_start_ind: int,
        tag_end_ind: int
        ) -> tuple[tuple[int, int], tuple[int, int]]: # The first tuple is `(a, b)` where `tokenized['input_ids'][a][b]` is the token corresponding to the start of the HTML tag's (raw) text. The second tuple is `(c, d)` where `tokenized['input_ids'][c][d]` is the token corresponding to the end of the HTML tag's (raw) text.
    start_seq = _search_seq_ind_for_char(tokenized['offset_mapping'], tag_start_ind)
    # start_index_in_seq = tokenized.char_to_token(batch_or_char_index=start_seq, char_index=tag_start_ind)
    start_index_in_seq = _search_within_seq_for_char(tokenized['offset_mapping'][start_seq], tag_start_ind)
    end_seq = _search_seq_ind_for_char(tokenized['offset_mapping'], tag_end_ind)
    # end_index_in_seq = tokenized.char_to_token(batch_or_char_index=end_seq, char_index=tag_end_ind)
    end_index_in_seq = _search_within_seq_for_char(tokenized['offset_mapping'][end_seq], tag_end_ind)
    return (start_seq, start_index_in_seq), (end_seq, end_index_in_seq)


def _min_max_char_ind_for_seq(
        offset_for_seq: list[tuple[int,int]] # An item in tokenized['offset_mapping']
        ):
    min_char_ind, max_char_ind = 0, 0
    for inds in offset_for_seq:
        if inds != (0,0):
            min_char_ind = inds[0]
            break
    for inds in reversed(offset_for_seq):
        if inds != (0,0):
            max_char_ind = inds[1]
            break
    return min_char_ind, max_char_ind

def _char_is_in_seq(
        offset_for_seq: list[int], # An item in tokenized['offset_mapping']
        char: int # The index of a character in the original raw text
        ) -> bool:
    min_char_ind, max_char_ind = _min_max_char_ind_for_seq(offset_for_seq)
    return min_char_ind <= char and char < max_char_ind

def _search_seq_ind_for_char(
        offsets: list[tuple[int, int]], # tokenized['offset_mapping']
        char: int # The index of a character in the original raw text
        ) -> int:
    """
    Binary search the index of the sequence containing the token at the 
    location of the index `char` within the original (raw) text.

    Based on pseudocode from https://pseudoeditor.com/guides/binary-search
    """
    left = 0
    right = len(offsets) - 1
    while left <= right:
        mid = (left + right) // 2
        min_char_ind, max_char_ind = _min_max_char_ind_for_seq(offsets[mid])
        if min_char_ind <= char and char < max_char_ind:
            return mid
        elif max_char_ind <= char:
            left = mid + 1
        else:
            right = mid - 1
    return -1  # This should not be returned under normal use.


def _search_within_seq_for_char(
        seq_offset: list[tuple[int, int]],
        char: int
    ) -> int:
    """
    Binary search for the index within the sequence corresponding
    to the token at the location of the index `char` within the
    original (raw) text.

    Based on pseudocode from https://pseudoeditor.com/guides/binary-search
    """
    left = 0
    right = len(seq_offset) - 1
    while left <= right:
        mid = (left + right) // 2
        min_char_ind, max_char_ind = seq_offset[mid] 
        if min_char_ind <= char and char < max_char_ind:
            return mid
        elif max_char_ind <= char:
            left = mid + 1
        else:
            right = mid - 1
    return -1  # This should not be returned under normal use.


def _set_ner_ids_for_tag(
        ner_ids: list[list[int]],
        start_seq: int, 
        start_index_in_seq: int,
        end_seq: int,
        end_index_in_seq: int,
        label2id: dict[str, int],
        ner_tag: str
        ) -> None:
    """
    After the locations of the tokens corresponding to a HTML tag have been found, 
    mark within `ner_ids` the appropriate NER tags at the locations corresponding
    to the tokens' locations.
    """
    ner_ids[start_seq][start_index_in_seq] = label2id[f"B-{ner_tag}"]
    i_ner_id = label2id[f"I-{ner_tag}"]
    seq, ind = start_seq, start_index_in_seq + 1
    while seq < end_seq or ind <= end_index_in_seq:
        if len(ner_ids[seq]) <= ind:
            seq += 1
            ind = 0
        else:
            ner_ids[seq][ind] = i_ner_id 
            ind += 1
    


def def_or_notat_from_html_tag(
        tag: bs4.element.Tag
        ) -> Union[str, None]:
    """
    Can be passed as the `ner_tag_from_html_tag` argument in `tokenize_html_data`
    for the purposes of compiling a dataset for definition and notation
    identification.

    The strings f"I-{output}" and f"B-{output}" are valid ner_tags. To use for 
    """
    if "definition" in tag.attrs:
        return "definition"
    elif "notation" in tag.attrs:
        return "notation"
    return None  # If the HTML tag carries neither definition nor notation data.

In [ ]:
#| hide
test_eq(_min_max_char_ind_for_seq([(0,0), (1,3), (3,4), (4,7), (7,15), (0,0)]), (1,15))

offsets = [[(0,0), (0,3), (4,5), (5,6), (6,7), (7,8), (8,9),],
           [(10,12), (13,14), (15,18), (18,24)],
           [(25,28), (29,35), (36,42), ]]
test_eq(_search_seq_ind_for_char(offsets, 0), 0)
test_eq(_search_seq_ind_for_char(offsets, 1), 0)
test_eq(_search_seq_ind_for_char(offsets, 5), 0)
test_eq(_search_seq_ind_for_char(offsets, 8), 0)
# I don't think that character index 9 is something that I need to worry about.
test_eq(_search_seq_ind_for_char(offsets, 10), 1)
test_eq(_search_seq_ind_for_char(offsets, 23), 1)
test_eq(_search_seq_ind_for_char(offsets, 25), 2)
test_eq(_search_seq_ind_for_char(offsets, 41), 2)

We continue with an example using the HTML data from the example for the `html_data_from_note` function.

In [ ]:
mf = MarkdownFile.from_string(
    r"""---
aliases: []
tags: []
---
# Galois group of a separable and normal finite field extension

Let $L/K$ be a separable and normal finite field extension. Its <b definition="Galois group of a separable and normal finite field extension">Galois group</b> <span notation="">$\operatorname{Gal}(L/K)$</span> is...

# Galois group of a separable and normal profinite field extension

In fact, the notion of a Galois group can be defined for profinite field extensions. Given a separable and normal profinite field extension $L/K$, say that
$L = \varinjlim_i L_i$ where $L_i/K$ are finite extensions. Its **Galois group** **$\operatorname{Gal}(L/K)$**

# See Also
# Meta
## References and Citations
""")

html_data = html_data_from_note(mf, vault=None, note_name=None)
print(html_data)

assert html_data['note_name'] is None

{'note_name': None, 'raw_text': 'Let $L/K$ be a separable and normal finite field extension. Its Galois group $\\operatorname{Gal}(L/K)$ is...\n\nIn fact, the notion of a Galois group can be defined for profinite field extensions. Given a separable and normal profinite field extension $L/K$, say that\n$L = \\varinjlim_i L_i$ where $L_i/K$ are finite extensions. Its Galois group $\\operatorname{Gal}(L/K)$\n', 'tags': [HTMLTagWithIndices(tag=<b definition="Galois group of a separable and normal finite field extension">Galois group</b>, start=64, end=76), HTMLTagWithIndices(tag=<span notation="">$\operatorname{Gal}(L/K)$</span>, start=77, end=102), HTMLTagWithIndices(tag=<b definition="">Galois group</b>, start=330, end=342), HTMLTagWithIndices(tag=<span notation="">$\operatorname{Gal}(L/K)$</span>, start=343, end=368)]}


In [ ]:
html_data['raw_text']
html_data["tags"]

[HTMLTagWithIndices(tag=<b definition="Galois group of a separable and normal finite field extension">Galois group</b>, start=64, end=76),
 HTMLTagWithIndices(tag=<span notation="">$\operatorname{Gal}(L/K)$</span>, start=77, end=102),
 HTMLTagWithIndices(tag=<b definition="">Galois group</b>, start=330, end=342),
 HTMLTagWithIndices(tag=<span notation="">$\operatorname{Gal}(L/K)$</span>, start=343, end=368)]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [ ]:
label2id = {
    "O": 0,
    "B-definition": 1,
    "I-definition": 2,
    "B-notation": 3,
    "I-notation": 4
}
tokens, ner_tag_ids = tokenize_html_data(html_data, tokenizer, 510, def_or_notat_from_html_tag, label2id)

For this example, `max_length` is set to 510 (tokens). The string ("Raw text") is not very long, so only one sequence should be present.

In [ ]:
test_eq(len(tokens), 1)
test_eq(len(ner_tag_ids), 1)

Now let us see what has been tagged:

In [ ]:
id2label = {value: key for key, value in label2id.items()}
id2label

{0: 'O',
 1: 'B-definition',
 2: 'I-definition',
 3: 'B-notation',
 4: 'I-notation'}

In [ ]:
for token, ner_tag in zip(tokens[0], ner_tag_ids[0]):
    if ner_tag != 0:
        print(f"{token}\t\t{id2label[ner_tag]}")

gal		B-definition
##ois		I-definition
group		I-definition
$		B-notation
\		I-notation
operator		I-notation
##name		I-notation
{		I-notation
gal		I-notation
}		I-notation
(		I-notation
l		I-notation
/		I-notation
k		I-notation
)		I-notation
$		I-notation
gal		B-definition
##ois		I-definition
group		I-definition
$		B-notation
\		I-notation
operator		I-notation
##name		I-notation
{		I-notation
gal		I-notation
}		I-notation
(		I-notation
l		I-notation
/		I-notation
k		I-notation
)		I-notation
$		I-notation


Let us set `max_length` to be shorter to observe an example of a tokenization of a single text across multiple sequences (Of course, in practice, the max token length would be set to be longer, say around 512 or 1024.):

In [ ]:
token_ids, ner_tag_ids = tokenize_html_data(html_data, tokenizer, 20, def_or_notat_from_html_tag, label2id)

In [ ]:
print(len(token_ids))
print(len(ner_tag_ids))

7
7


In [ ]:
ner_tag_ids

[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2],
 [2, 2, 2, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 2, 3, 4, 4],
 [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 0]]

## Parse Data from standalone text

Alternatively, data can be gathered from a str with definition and notations marked with some general formatting, as long as a parser for those markings is specified. 

In [ ]:
#| export
def _calculate_clean_indices_and_create_tags(
        original_text: str,
        markings: list[tuple[str, int, int, dict]]
        ) -> tuple[str, list[HTMLTagWithIndices]]:
    """Helper: returns (clean_text, list_of_tags)."""
    results = []
    clean_text_parts = []
    current_raw_idx = 0
    clean_text_len = 0
    soup = bs4.BeautifulSoup("", 'html.parser')

    for content, raw_start, raw_end, attrs in markings:
        # Append text BEFORE the mark
        pre_text = original_text[current_raw_idx:raw_start]
        clean_text_parts.append(pre_text)
        
        pre_text_len = len(pre_text)
        clean_text_len += pre_text_len
        
        # Calculate indices
        tag_start = clean_text_len
        tag_end = tag_start + len(content)
        
        # Create Tag
        tag_name = "b" if "definition" in attrs else "span"
        tag = soup.new_tag(tag_name, **attrs)
        tag.string = content
        results.append(HTMLTagWithIndices(tag, tag_start, tag_end))
        
        # Append the CONTENT of the mark (clean text)
        clean_text_parts.append(content)
        
        clean_text_len += len(content)
        current_raw_idx = raw_end

    # Append remaining text
    clean_text_parts.append(original_text[current_raw_idx:])
    
    return "".join(clean_text_parts), results

In [ ]:
#| export
def extract_html_tag_indices_from_marked_text(
        text: str, # The text containing custom markings (e.g. "Let [NOT:G] be a [DEF:group]").
        marker_parser: Callable[[str], list[tuple[str, int, int, dict]]] # A function that parses the text and returns a list of tuples. Each tuple should contain: (1) The *inner content* of the marked section, (2) The *start index* of the marking in `text`, (3) The *end index* of the marking in `text`, and (4) A dictionary of *attributes* for the HTML tag.
        ) -> list[HTMLTagWithIndices]: # A list of `HTMLTagWithIndices` objects. The start/end indices corresponds to the *inner content's* location in a "clean" version of the text (where markings are removed).
    """
    Extracts a list of HTML tags and their indices from marked text.
    """
    markings = marker_parser(text)
    markings.sort(key=lambda x: x[1])
    _, tags = _calculate_clean_indices_and_create_tags(text, markings)
    return tags

In [ ]:
import re

def bracket_parser(text: str) -> list[tuple[str, int, int, dict]]:
    """
    Parses marks like [DEF:content] and [NOT:content].
    Returns list of (content, start, end, attrs).
    """
    # Regex to find [TYPE:content]
    # Note: Using non-greedy match (.*?) to handle multiple brackets correctly
    pattern = re.compile(r'\[(DEF|NOT):(.*?)]')
    
    results = []
    for match in pattern.finditer(text):
        tag_type = match.group(1)
        content = match.group(2)
        start, end = match.span()
        
        attrs = {}
        if tag_type == 'DEF':
            attrs['definition'] = ''
        else:
            attrs['notation'] = ''
            
        results.append((content, start, end, attrs))
    return results

# --- Test Setup ---
# Marked Text: "The [DEF:Galois group] [NOT:$\operatorname{Gal}(L/K)$] is..."
marked_text = r"The [DEF:Galois group] [NOT:$\operatorname{Gal}(L/K)$] is..."

# "Clean" Text (what indices refer to): 
# "The Galois group $\operatorname{Gal}(L/K)$ is..."
# Indices Breakdown:
# "The " -> 0-4
# "Galois group" -> 4-16 (Length 12)
# " " -> 16-17
# "$\operatorname{Gal}(L/K)$" -> 17-42 (Length 25)
# " is..." -> 42...

tag_data = extract_html_tag_indices_from_marked_text(marked_text, bracket_parser)

# --- Assertions ---
assert len(tag_data) == 2, f"Expected 2 tags, got {len(tag_data)}"

# 1. Check Definition Tag ("Galois group")
defn = tag_data[0]
assert defn.tag.string == "Galois group"
assert "definition" in defn.tag.attrs
assert defn.start == 4
assert defn.end == 16, f"Expected end 16, got {defn.end}"

# 2. Check Notation Tag ("$\operatorname{Gal}(L/K)$")
notat = tag_data[1]
expected_math = r"$\operatorname{Gal}(L/K)$"
assert notat.tag.string == expected_math
assert "notation" in notat.tag.attrs
assert notat.start == 17, f"Expected start 17, got {notat.start}"
assert notat.end == 42, f"Expected end 42, got {notat.end}"

print("html_tag_data_from_marked_text (Galois example) passed!")

html_tag_data_from_marked_text (Galois example) passed!


In [ ]:
#| export
def html_data_from_marked_text(
        text: str, # The text containing custom markings (e.g. "Let [NOT:G] be a [DEF:group]").
        marker_parser: Callable[[str], list[tuple[str, int, int, dict]]] # A function that parses the text and returns a list of tuples. Each tuple should contain: (1) The *inner content* of the marked section, (2) The *start index* of the marking in `text`, (3) The *end index* of the marking in `text`, and (4) A dictionary of *attributes* for the HTML tag.
        ) -> StrAndHTMLTagsWithIndices: # An object containing the "clean" text (markings removed) and the list of HTML tags with their indices in that clean text.
    """
    Creates an `StrAndHTMLTagsWithIndices` object from text with custom markings.
    """
    markings = marker_parser(text)
    markings.sort(key=lambda x: x[1])
    clean_text, tags = _calculate_clean_indices_and_create_tags(text, markings)
    
    return StrAndHTMLTagsWithIndices(clean_text, tags)

In [ ]:
#| hide
import re

def bracket_parser(text: str) -> list[tuple[str, int, int, dict]]:
    """Parses [DEF:content] and [NOT:content]."""
    pattern = re.compile(r'\[(DEF|NOT):(.*?)]')
    results = []
    for match in pattern.finditer(text):
        tag_type = match.group(1)
        content = match.group(2)
        start, end = match.span()
        attrs = {'definition' if tag_type == 'DEF' else 'notation': ''}
        results.append((content, start, end, attrs))
    return results

# --- Test Setup ---
# Marked Text: "The [DEF:Galois group] [NOT:$\operatorname{Gal}(L/K)$] is..."
marked_text = r"The [DEF:Galois group] [NOT:$\operatorname{Gal}(L/K)$] is..."

# Expected "Clean" Text (markings stripped, content kept)
expected_clean_text = r"The Galois group $\operatorname{Gal}(L/K)$ is..."

# --- Run Function ---
data_obj = html_data_from_marked_text(marked_text, bracket_parser)

print(data_obj)
# --- Assertions ---

# 1. Check Clean Text
assert data_obj.raw_text == expected_clean_text, \
    f"Clean text mismatch.\nExpected: '{expected_clean_text}'\nGot:      '{data_obj.raw_text}'"

# 2. Check Tag Count
assert len(data_obj.tags) == 2, f"Expected 2 tags, got {len(data_obj.tags)}"

# 3. Check Definition Tag ("Galois group")
# Indices in clean text: "The " (4) -> start 4
# "Galois group" (length 12) -> end 16
defn = data_obj.tags[0]
assert defn.tag.string == "Galois group"
assert "definition" in defn.tag.attrs
assert defn.start == 4
assert defn.end == 16

# 4. Check Notation Tag ("$\operatorname{Gal}(L/K)$")
# Indices in clean text:
# "The Galois group " (17 chars: 4 + 12 + 1 space) -> start 17?
# Wait: "The " (4) + "Galois group" (12) + " " (1) = 17.
# So Notation starts at 17.
# Content: "$\operatorname{Gal}(L/K)$" (Length 25)
# End: 17 + 25 = 42.
notat = data_obj.tags[1]
expected_math = r"$\operatorname{Gal}(L/K)$"
assert notat.tag.string == expected_math
assert "notation" in notat.tag.attrs
assert notat.start == 17
assert notat.end == 42

print("html_data_from_marked_text (Galois example) passed!")

StrAndHTMLTagsWithIndices(raw_text='The Galois group $\\operatorname{Gal}(L/K)$ is...', tags=[HTMLTagWithIndices(tag=<b definition="">Galois group</b>, start=4, end=16), HTMLTagWithIndices(tag=<span notation="">$\operatorname{Gal}(L/K)$</span>, start=17, end=42)])
html_data_from_marked_text (Galois example) passed!


### Miscellaneous parse formatting

The following is intended to parse latex code of the author's writing for definition and notation markings

In [ ]:
#| export
def latex_highlight_parser(text: str) -> list[tuple[str, int, int, dict]]:
    """
    Parses LaTeX highlighting commands (\\hldef, \\hl, \\hlin, \\hlalign) to identify
    definitions and notations.
    
    This function is designed to be passed as the `marker_parser` argument to
    `html_data_from_marked_text`.

    It handles nested braces using recursive regex and strips surrounding whitespace 
    from the inner content (e.g., stripping padding newlines in `\\hlalign{\\n ... \\n}`).
    It also implements special logic for `\\hlin` to expand the notation range to 
    include surrounding `$$` delimiters if present.

    Returns
    -------
    list[tuple[str, int, int, dict]]
        A list of tuples representing the found markings. Each tuple contains:
        1. **Content** (`str`): The inner text of the marking (stripped of padding whitespace).
           This is the text that will remain in the "clean" output and be wrapped by the tag.
        2. **Start Index** (`int`): The start index of the *entire marking wrapper* 
           (e.g., the index of `\\`) in the original `text`.
        3. **End Index** (`int`): The end index of the *entire marking wrapper* 
           (e.g., the index after `}`) in the original `text`.
        4. **Attributes** (`dict`): A dictionary of HTML attributes, e.g. 
           `{'definition': ''}` or `{'notation': ''}`.
    """
    pattern = r'\\(hldef|hlalign|hlin|hl)\s*(\{(?:[^{}]++|(?2))*\})'
    results = []
    
    for match in regex.finditer(pattern, text):
        cmd_type = match.group(1)
        full_brace_group = match.group(2)
        raw_inner = full_brace_group[1:-1]
        match_start, match_end = match.span()
        
        # Strip padding whitespace from content
        lstripped = raw_inner.lstrip()
        stripped = lstripped.rstrip()
        content = stripped
        
        start = match_start
        end = match_end
        attrs = {'definition' if cmd_type == 'hldef' else 'notation': ''}
        
        if cmd_type == 'hlin':
            # 1. Scan Backwards for start $$
            p = start - 1
            while p >= 0 and text[p].isspace(): p -= 1
            
            # Check if we hit $$ immediately (ignoring space)
            if p >= 1 and text[p] == '$' and text[p-1] == '$':
                found_start_dollars = True
                new_start = p - 1
                # BUG WAS HERE: prefix = "" 
                # FIX: Capture the text between $$ (p+1) and \hlin (start)
                prefix = text[p+1 : start] 
            else:
                found_start_dollars = False
                prefix = "" # Initialize for safety

            
            # 2. Scan Forwards for end $$
            # We want to allow punctuation like "." or "," between } and $$
            # E.g. $$ \hlin{x}. $$
            
            q = end
            suffix = ""
            found_end_dollars = False
            
            # Heuristic: Scan forward for a limited distance or until $$
            # We capture everything between } and $$ into 'suffix'
            # Stop if we hit a newline (safeguard)
            temp_q = q
            while temp_q < len(text) - 1:
                if text[temp_q] == '\n': break
                
                if text[temp_q] == '$' and text[temp_q+1] == '$':
                    found_end_dollars = True
                    new_end = temp_q + 2
                    # The text between original end and $$ is the suffix
                    suffix = text[end:temp_q]
                    # Clean the suffix? Usually we just want to include it.
                    # e.g. suffix might be ". " (period and space)
                    # We usually trim the space before the $$, but keeping it is safer for fidelity.
                    break
                temp_q += 1
            
            # Only apply expansion if we found BOTH start and end $$
            # AND (optionally) if we found the start $$ directly. 
            # (Does it make sense to have content BEFORE \hlin? e.g. $$ x = \hlin{y} $$?
            #  If so, we should scan backwards for $$ similarly to how we scanned forwards).
            
            # Let's implement symmetric scanning for robustness.
            
            # RE-TRY Backward Scan with content capture
            if not found_start_dollars:
                 temp_p = start - 1
                 while temp_p >= 1:
                     if text[temp_p] == '\n': break
                     if text[temp_p] == '$' and text[temp_p-1] == '$':
                         found_start_dollars = True
                         new_start = temp_p - 1
                         prefix = text[temp_p+1 : start] # Content between $$ and \hlin
                         break
                     temp_p -= 1

            if found_start_dollars and found_end_dollars:
                # We found $$ ... \hlin{...} ... $$
                # We want the tag to cover the WHOLE thing: "$$ prefix content suffix $$"
                # And we want to remove the WHOLE thing from the text.
                
                # Careful: The 'prefix' and 'suffix' currently contain the raw characters 
                # from the marked text (including potentially ignored spaces).
                # We should probably strip excessive padding next to the $$ inside the tag?
                # Standard convention: $$ content $$ -> Tag content "$$ content $$"
                
                # Let's just assemble it raw to preserve user's punctuation/spacing logic,
                # then maybe strip outer edges if needed.
                
                # Current 'content' is stripped inner content of \hlin.
                # 'prefix' is text between $$ and \hlin.
                # 'suffix' is text between \hlin and $$.
                
                # Example: $$ \hlin{K}. $$
                # prefix = " "
                # content = "K"
                # suffix = ". "
                # Result: "$$ K. $$"
                
                full_content = f"$${prefix}{content}{suffix}$$"
                
                content = full_content
                start = new_start
                end = new_end

        results.append((content, start, end, attrs))
        
    return results
    # pattern = r'\\(hldef|hlalign|hlin|hl)\s*(\{(?:[^{}]++|(?2))*\})'
    
    # results = []
    
    # for match in regex.finditer(pattern, text):
    #     cmd_type = match.group(1)
    #     full_brace_group = match.group(2)
        
    #     # Raw content inside braces (e.g. " \n Content \n ")
    #     raw_inner = full_brace_group[1:-1]
        
    #     # Calculate indices relative to the 'text' string
    #     # Match span covers `\hl{...}`
    #     match_start, match_end = match.span()
        
    #     # We want to identify where the "real" content starts/ends inside the match
    #     # Start of raw_inner is: match_end - 1 (closing brace) - len(raw_inner)
    #     # Actually easier: find start of {
    #     brace_start_idx = match.start(2) # Index of {
    #     inner_start_idx = brace_start_idx + 1
        
    #     # Find leading/trailing whitespace length
    #     lstripped = raw_inner.lstrip()
    #     leading_ws_len = len(raw_inner) - len(lstripped)
        
    #     stripped = lstripped.rstrip()
    #     trailing_ws_len = len(lstripped) - len(stripped)
        
    #     content = stripped
    #     attrs = {'definition' if cmd_type == 'hldef' else 'notation': ''}
        
    #     # 1. Base Range: The command wrapper `\cmd{` and `}`.
    #     # We want to effectively say:
    #     # "Remove `\cmd{ \n`", keep `Content`, "Remove `\n }`".
    #     # But our interface only supports "Remove `Range`, Insert `Content`".
        
    #     # If we return `start=match_start`, `end=match_end`, `content=stripped`:
    #     # "Remove `\hl{ \n Content \n }`. Insert `Content`."
    #     # Result Clean Text: `Content`. (Newlines LOST).
        
    #     # If the user WANTS those newlines preserved in the clean text (outside the tag),
    #     # we have a problem. The current architecture assumes "Marked Region" -> "Tag".
    #     # It doesn't support "Marked Region" -> "Prefix + Tag + Suffix".
        
    #     # DECISION: 
    #     # For `\hlalign`, users usually write:
    #     # \hlalign{
    #     # \begin{align}
    #     # ...
    #     # \end{align}
    #     # }
    #     # They EXPECT the clean text to contain the newlines so the align renders correctly?
    #     # Actually, LaTeX doesn't care about the surrounding newlines much.
    #     # `\begin{align}...\end{align}` is valid without extra newlines.
    #     # So losing the newlines inside the braces is probably ACCEPTABLE and cleaner.
        
    #     # HOWEVER, if `$$ \hlin{ x } $$` -> `$$x$$`. Losing spaces is fine.
        
    #     # Special Logic for \hlin ($$) remains...
        
    #     # Let's apply the stripping logic:
        
    #     start = match_start
    #     end = match_end
        
    #     if cmd_type == 'hlin':
    #         # ... (Logic to expand to $$ remains same, but apply to `stripped` content) ...
    #         # Re-implementing simplified logic for clarity in this snippet:
    #         p = start - 1
    #         while p >= 0 and text[p].isspace(): p -= 1
    #         if p >= 1 and text[p] == '$' and text[p-1] == '$':
    #             new_start = p - 1
    #             q = end
    #             while q < len(text) and text[q].isspace(): q += 1
    #             if q < len(text) - 1 and text[q] == '$' and text[q+1] == '$':
    #                 new_end = q + 2
    #                 content = f"$${content}$$"
    #                 start = new_start
    #                 end = new_end

    #     results.append((content, start, end, attrs))
        
    # return results

In [ ]:
from bs4 import BeautifulSoup

# --- Setup ---
# Input Text with mixed highlighting
# 1. Definition: \hldef{Galois group}
# 2. Inline Notation: \hl{$\operatorname{Gal}(L/K)$}
# 3. Display Notation: $$\hlin{x^2}$$
marked_text = r"The \hldef{Galois group} \hl{$\operatorname{Gal}(L/K)$} is defined. Display: $$\hlin{x^2}$$"

# Expected Clean Text
expected_clean = r"The Galois group $\operatorname{Gal}(L/K)$ is defined. Display: $$x^2$$"

# --- Run ---
data_obj = html_data_from_marked_text(marked_text, latex_highlight_parser)

# --- Assertions ---

# 1. Check Clean Text
assert data_obj.raw_text == expected_clean, \
    f"Clean text mismatch.\nExpected: {expected_clean}\nGot:      {data_obj.raw_text}"

# 2. Check Tag Count (3 tags)
assert len(data_obj.tags) == 3

# 3. Check Definition
defn = data_obj.tags[0]
assert defn.tag.string == "Galois group"
assert "definition" in defn.tag.attrs
# "The " (4) -> start 4
assert defn.start == 4

# 4. Check Inline Notation
notat_inline = data_obj.tags[1]
assert notat_inline.tag.string == r"$\operatorname{Gal}(L/K)$"
assert "notation" in notat_inline.tag.attrs
# "The Galois group " (17) -> start 17
assert notat_inline.start == 17

# 5. Check Display Notation
# Clean text segment: " is defined. Display: $$"
# "The Galois group $\operatorname{Gal}(L/K)$" (length 17 + 25 = 42)
# " is defined. Display: $$" (length 22)
# Start index should be 42 + 22 = 64
notat_display = data_obj.tags[2]
assert notat_display.tag.string == "$$x^2$$"
assert "notation" in notat_display.tag.attrs

print("latex_highlight_parser test passed!")

latex_highlight_parser test passed!


In [ ]:
print(data_obj.raw_text)
print(data_obj.tags)

The Galois group $\operatorname{Gal}(L/K)$ is defined. Display: $$x^2$$
[HTMLTagWithIndices(tag=<b definition="">Galois group</b>, start=4, end=16), HTMLTagWithIndices(tag=<span notation="">$\operatorname{Gal}(L/K)$</span>, start=17, end=42), HTMLTagWithIndices(tag=<span notation="">$$x^2$$</span>, start=64, end=71)]


In [ ]:

# --- Setup ---
# 1. Nested Braces: \hl{$\{x\}$}
# 2. Display Math: $$\hlin{x^2}$$
# 3. Align: \hlalign{\begin{align}a&=b\end{align}}

marked_text = (
    r"Set \hl{$\{x\}$}. "
    r"Display: $$\hlin{x^2}$$ "
    r"Align: \hlalign{\begin{align}a&=b\end{align}}"
)

# Expected Clean Text
# 1. $\begin{Bmatrix}x\end{Bmatrix}$ (restored)
# 2. $$x^2$$ (restored, tag wraps entire thing)
# 3. \begin{align}a&=b\end{align} (restored)
expected_clean = (
    r"Set $\{x\}$. "
    r"Display: $$x^2$$ "
    r"Align: \begin{align}a&=b\end{align}"
)

# --- Run ---
data_obj = html_data_from_marked_text(marked_text, latex_highlight_parser)

# --- Assertions ---
# 1. Clean Text
assert data_obj.raw_text == expected_clean, \
    f"Clean text mismatch.\nExpected: '{expected_clean}'\nGot:      '{data_obj.raw_text}'"

# 2. Nested Brace Tag
tag_nested = data_obj.tags[0]
assert tag_nested.tag.string == r"$\{x\}$"
assert "notation" in tag_nested.tag.attrs

# 3. Display Math Tag
# Should wrap "$$x^2$$" because parser expanded to include $$
tag_display = data_obj.tags[1]
assert tag_display.tag.string == r"$$x^2$$"
assert "notation" in tag_display.tag.attrs

# 4. Align Tag
tag_align = data_obj.tags[2]
assert tag_align.tag.string == r"\begin{align}a&=b\end{align}"
assert "notation" in tag_align.tag.attrs

print("Robust parser tests passed!")
print(data_obj)

Robust parser tests passed!
StrAndHTMLTagsWithIndices(raw_text='Set $\\{x\\}$. Display: $$x^2$$ Align: \\begin{align}a&=b\\end{align}', tags=[HTMLTagWithIndices(tag=<span notation="">$\{x\}$</span>, start=4, end=11), HTMLTagWithIndices(tag=<span notation="">$$x^2$$</span>, start=22, end=29), HTMLTagWithIndices(tag=<span notation="">\begin{align}a&amp;=b\end{align}</span>, start=37, end=65)])


In [ ]:
#| hide
#| notest

# --- Setup ---
# Input has padding newlines inside the braces:
# {
#   \begin{align} ... \end{align}
# }
marked_text = r"""
Here is some text.
\hlalign{
\begin{align}
asdf
\end{align}
}
End of text.
"""

# Expected Clean Text:
# The `\hlalign{` and `}` are removed.
# Crucially, the padding newlines inside the braces are ALSO removed by the strip() logic.
# So we expect NO newline between "Here is some text." and "\begin{align}",
# unless there was one outside the command (which there is: line 2 has a newline).
#
# Original:
# Line 1: "Here is some text."
# Line 2: "\hlalign{" -> Replaced by "\begin{align}" (start of content)
# ... content ...
# Line 6: "}" -> Replaced by "" (end of content)
# Line 7: "End of text."

# So the clean text will effectively collapse the structure slightly.
# "Here is some text.\n\begin{align}\nasdf\n\end{align}\nEnd of text."
expected_clean = (
    "Here is some text.\n"
    r"\begin{align}" + "\n"
    r"asdf" + "\n"
    r"\end{align}" + "\n"
    "End of text."
)

# --- Run ---
data_obj = html_data_from_marked_text(marked_text, latex_highlight_parser)

# --- Assertions ---

# 1. Check Clean Text
# If the parser failed to strip padding, we would see extra newlines (e.g. \n\n\begin...)
assert data_obj.raw_text.strip() == expected_clean.strip(), \
    f"Clean text mismatch.\nExpected:\n{repr(expected_clean)}\nGot:\n{repr(data_obj.raw_text)}"

# 2. Check Tag Content
tag_align = data_obj.tags[0]
tag_content = tag_align.tag.string

# Verify NO leading/trailing newlines in the tag content itself
# (It should start immediately with \begin and end with \end)
assert tag_content.startswith(r"\begin{align}"), f"Tag content has leading garbage: {repr(tag_content[:20])}"
assert tag_content.endswith(r"\end{align}"), f"Tag content has trailing garbage: {repr(tag_content[-20:])}"

# Verify INTERNAL newlines are preserved (between begin and asdf)
assert "\nasdf\n" in tag_content, "Internal newlines were incorrectly stripped!"

print("Multiline hlalign padding test passed!")

Multiline hlalign padding test passed!


Here are some examples of `latex_highlight_parser` in the context of `html_data_from_marked_text`

In [ ]:
# from trouver.machinelearning.tokenize.def_and_notat_token_classification import (
#     html_data_from_marked_text, 
#     latex_highlight_parser
# )

# Input text with marking commands
marked_text = r"The \hldef{Galois group} \hl{$\operatorname{Gal}(L/K)$} is defined."

# Process the text
data = html_data_from_marked_text(marked_text, latex_highlight_parser)

# 1. Clean Text Verification
# The commands \hldef{...} and \hl{...} are stripped, leaving only the content.
expected_clean = r"The Galois group $\operatorname{Gal}(L/K)$ is defined."
assert data.raw_text == expected_clean

# 2. Tag Verification
# Tag 1: Definition
def_tag = data.tags[0]
assert def_tag.tag.string == "Galois group"
assert "definition" in def_tag.tag.attrs
# "The " is 4 chars long, so start is 4.
assert def_tag.start == 4 
assert def_tag.end == 4 + len("Galois group")

# Tag 2: Notation
not_tag = data.tags[1]
assert not_tag.tag.string == r"$\operatorname{Gal}(L/K)$"
assert "notation" in not_tag.tag.attrs
# Starts after "The Galois group " (17 chars)
assert not_tag.start == 17

print("Example 1 Passed")

Example 1 Passed


In [ ]:
# Input: Display math where \hlin wraps the inner equation
# Note: The parser handles the spaces around the `\hlin` command inside the $$
marked_text = r"The kernel is trivial: $$ \hlin{ K = \{e\} } $$"

data = html_data_from_marked_text(marked_text, latex_highlight_parser)

# 1. Clean Text Verification
# The result should look like standard LaTeX display math.
# The spaces inside the braces are stripped by the parser, but the spaces
# between $$ and the content depend on how we reconstruct it.
# Our parser returns content "$$K = \{e\}$$" (with $$ added).
# And it consumes the original "$$ \hlin{...} $$".
# So the clean text is just the content inserted at that spot.
expected_clean = r"The kernel is trivial: $$ K = \{e\} $$"
assert data.raw_text == expected_clean

# 2. Tag Verification
tag = data.tags[0]
# The tag should wrap the WHOLE math block, including $$
assert tag.tag.string == r"$$ K = \{e\} $$"
assert "notation" in tag.tag.attrs

print("Example 2 Passed")

Example 2 Passed


In [ ]:
#| hide

# --- Setup ---
# Case: Punctuation inside $$ but outside \hlin
marked_text = r"The kernel is trivial: $$ \hlin{ K = \{e\} }. $$"

data = html_data_from_marked_text(marked_text, latex_highlight_parser)

# 1. Clean Text Verification
# Original text logic would normally produce:
# "The kernel is trivial: $$ K = \{e\} . $$" (spaces preserved from prefix/suffix)
#
# Let's trace the parser:
# prefix = " " (between $$ and \hlin)
# content = "K = \{e\}"
# suffix = ". " (between } and $$)
# result = "$$ K = \{e\}. $$"
expected_clean = r"The kernel is trivial: $$ K = \{e\}. $$"

assert data.raw_text == expected_clean, \
    f"Clean Text Mismatch.\nExp: '{expected_clean}'\nGot: '{data.raw_text}'"

# 2. Tag Verification
tag = data.tags[0]
tag_str = tag.tag.string

# Ensure the tag wraps the whole $$ block
assert tag_str.startswith("$$")
assert tag_str.endswith("$$")
assert r"K = \{e\}" in tag_str
# Ensure the period is INSIDE the tag
assert "." in tag_str
# Ensure the tag covers the full range
assert tag.start == 23 # Index of $$ start in "The kernel is trivial: "

print("Typo/Punctuation inside $$ test passed!")

Typo/Punctuation inside $$ test passed!


In [ ]:
#| hide
# Input: Multi-line align block with padding newlines inside the command
marked_text = r"""Consider the sequence:
\hlalign{
\begin{align}
0 \to A \to B \to 0
\end{align}
}
It is exact."""

data = html_data_from_marked_text(marked_text, latex_highlight_parser)

# 1. Clean Text Verification
# Padding newlines inside \hlalign{...} are stripped.
# The newline AFTER "sequence:" remains.
# The newline BEFORE "It is exact" remains.
expected_clean = r"""Consider the sequence:
\begin{align}
0 \to A \to B \to 0
\end{align}
It is exact."""

# Strip checks to ignore potential leading/trailing whitespace difference in the file itself
assert data.raw_text.strip() == expected_clean.strip()

# 2. Tag Verification
tag = data.tags[0]
content = tag.tag.string

# Ensure the tag content preserves the internal structure
assert r"\begin{align}" in content
assert r"0 \to A" in content
assert r"\end{align}" in content
assert content.startswith(r"\begin{align}")
assert content.endswith(r"\end{align}")
# Ensure internal newlines are preserved
assert "\n" in content
# Ensure tag is marked as notation
assert "notation" in tag.tag.attrs

print("Example 3 Passed")

Example 3 Passed


## Augmenting data

In [ ]:
#| export
def _split_text_by_html_data_parts(
        # text_tags_and_locations = StrAndHTMLTagsWithIndices
        datapoint: HTMLData
        ) -> list[tuple[str, Union[bs4.element.Tag, None]]]:
    r"""
    Helper function to `augment_html_data`.
    """
    to_return: list[tuple[str, Union[bs4.element.Tag, None]]] = []
    split_points: list[int] = []
    for tag_ind in datapoint['tags']:
        split_points.append(tag_ind.start)
        split_points.append(tag_ind.end)
    pieces: list[str] = split_string_at_indices(datapoint['raw_text'], split_points)
    for i, piece in enumerate(pieces):
        if i % 2 == 0:
            to_return.append((piece, None))
        else:
            to_return.append((piece, datapoint['tags'][int(i/2)].tag))
    return to_return

In [ ]:
#|hide
sample_text = r"""# Galois group of a separable and normal finite field extension

Let $L/K$ be a separable and normal finite field extension. Its <b definition="Galois group of a separable and normal finite field extension">Galois group</b> <span notation="">$\operatorname{Gal}(L/K)$</span> is...

# See Also
# Meta
## References and Citations
"""

sample_str_and_html_tags_with_indices: StrAndHTMLTagsWithIndices = remove_html_tags_in_text(sample_text)
sample_html_data = HTMLData(note_name=None, raw_text=sample_str_and_html_tags_with_indices.raw_text, tags=sample_str_and_html_tags_with_indices.tags)
output = _split_text_by_html_data_parts(sample_html_data)
assert isinstance(output[1][1], bs4.element.Tag)
output

[('# Galois group of a separable and normal finite field extension\n\nLet $L/K$ be a separable and normal finite field extension. Its ',
  None),
 ('Galois group',
  <b definition="Galois group of a separable and normal finite field extension">Galois group</b>),
 (' ', None),
 ('$\\operatorname{Gal}(L/K)$',
  <span notation="">$\operatorname{Gal}(L/K)$</span>),
 (' is...\n\n# See Also\n# Meta\n## References and Citations\n', None)]

In [ ]:
#| hide
sample_text = r"""<b definition="">Hello</b> asdf <b notation=""> $\operatorname{Gal}$</b>"""
sample_str_and_html_tags_with_indices: StrAndHTMLTagsWithIndices = remove_html_tags_in_text(sample_text)
sample_html_data = HTMLData(note_name=None, raw_text=sample_str_and_html_tags_with_indices.raw_text, tags=sample_str_and_html_tags_with_indices.tags)
output = _split_text_by_html_data_parts(sample_html_data)
assert isinstance(output[1][1], bs4.element.Tag)
assert isinstance(output[3][1], bs4.element.Tag)
test_is(output[4][1], None)
output

[('', None),
 ('Hello', <b definition="">Hello</b>),
 (' asdf ', None),
 (' $\\operatorname{Gal}$', <b notation=""> $\operatorname{Gal}$</b>),
 ('', None)]

In [ ]:
print(sample_str_and_html_tags_with_indices.tags)

[HTMLTagWithIndices(tag=<b definition="">Hello</b>, start=0, end=5), HTMLTagWithIndices(tag=<b notation=""> $\operatorname{Gal}$</b>, start=11, end=32)]


In [ ]:
#| export
def augment_html_data(
        datapoint: HTMLData,
        num_augmentation_sets: int = 1, # Each augmentation set consists of an augmentation with low, medium, and high probability modifications.
        seed: Optional[int] = None
        ) -> list[HTMLData]:
    r"""Augment a given datapoint for HTML tagging.

    """
    augmented_datapoints: list[HTMLData] = []
    pieces: list[tuple[str, Union[bs4.element.Tag, None]]] = _split_text_by_html_data_parts(
        datapoint)
    if seed is not None:
        random.seed(seed)
    for _ in range(num_augmentation_sets):
        augmented_datapoints.append(
            _augment_html_data_once(pieces, 'low', datapoint['note_name']))
        augmented_datapoints.append(
            _augment_html_data_once(pieces, 'mid', datapoint['note_name']))
        augmented_datapoints.append(
            _augment_html_data_once(pieces, 'hi', datapoint['note_name']))
        # augmented_datapoints.append(_augment_html_data_once(pieces, 'high'))
    return augmented_datapoints


def _augment_html_data_once(
        pieces: list[tuple[str, Union[bs4.element.Tag, None]]],
        modification: Literal['low', 'mid', 'high'],
        note_name: str,
        ) -> HTMLData:
    methods = [
        # (push_dollar_signs,0.2),
        (remove_font_styles_at_random, 0.1), (change_font_styles_at_random, 0.2), (change_greek_letters_at_random, 0.1), 
        (remove_math_keywords,0.1), (random_latex_command_removal,0.2),
        (random_word_removal,0.1), (dollar_sign_manipulation,0.05),
        (random_char_modification,0.001)]
    if modification == 'low':
        method_inclusion_chance = 0.3
        scale = 0.5
    elif modification == 'mid':
        method_inclusion_chance = 0.5
        scale = 1.0
    else:
        method_inclusion_chance = 0.8
        scale = 1.5
    
    random_methods = []
    def create_method(method, p, scale):
        return lambda x: method(x, p=p*scale)
    for method, p in methods:
        if random.random() < method_inclusion_chance:
            random_methods.append(create_method(method, p, scale))
    # random_methods = [
    #     lambda x: method(x, p=p*scale) for method, p in methods if random.random() < method_inclusion_chance]
    augmented_pieces = [
        (augment_text(text, random_methods), copy.copy(tag))
        for text, tag in pieces]
    accumulated_len: int = 0
    accumulated_text: str = ""
    tags_with_indices: list[HTMLTagWithIndices] = []
    for text, tag in augmented_pieces:
        accumulated_text = f'{accumulated_text}{text}'
        if tag:
            tag.string = text
            tags_with_indices.append(HTMLTagWithIndices(
                tag, accumulated_len, accumulated_len + len(text)))
        accumulated_len += len(text)
    return HTMLData(note_name=note_name, raw_text=accumulated_text, tags=tags_with_indices)

The `augment_html_data` can be used to augment definition and notation token classification data gathered via `html_data_from_note`.

In [ ]:
mf = MarkdownFile.from_string(
    r"""---
aliases: []
tags: []
---
# Galois group of a separable and normal finite field extension

Let $L/K$ be a separable and normal finite field extension. Its <b definition="Galois group of a separable and normal finite field extension">Galois group</b> <span notation="">$\operatorname{Gal}(L/K)$</span> is...

# Galois group of a separable and normal profinite field extension

In fact, the notion of a Galois group can be defined for profinite field extensions. Given a separable and normal profinite field extension $L/K$, say that
$L = \varinjlim_i L_i$ where $L_i/K$ are finite extensions. Its **Galois group** **$\operatorname{Gal}(L/K)$**

# See Also
# Meta
## References and Citations
""")

html_data: HTMLData = html_data_from_note(mf)
output = augment_html_data(html_data,seed=None)
html_data, output
# html_data

({'note_name': None,
  'raw_text': 'Let $L/K$ be a separable and normal finite field extension. Its Galois group $\\operatorname{Gal}(L/K)$ is...\n\nIn fact, the notion of a Galois group can be defined for profinite field extensions. Given a separable and normal profinite field extension $L/K$, say that\n$L = \\varinjlim_i L_i$ where $L_i/K$ are finite extensions. Its Galois group $\\operatorname{Gal}(L/K)$\n',
  'tags': [HTMLTagWithIndices(tag=<b definition="Galois group of a separable and normal finite field extension">Galois group</b>, start=64, end=76),
   HTMLTagWithIndices(tag=<span notation="">$\operatorname{Gal}(L/K)$</span>, start=77, end=102),
   HTMLTagWithIndices(tag=<b definition="">Galois group</b>, start=330, end=342),
   HTMLTagWithIndices(tag=<span notation="">$\operatorname{Gal}(L/K)$</span>, start=343, end=368)]},
 [{'note_name': None,
   'raw_text': 'Let $L/K$ be a separable and normal finite field extension. Its Galois group \\operatorname{Gal}(L/K) is...\n\nIn fac

## Gathering data 

The following is sample code to then gather data for definition/notation identification

In [ ]:
#| notest

# TODO: test

notes = [] # Replace with actual notes
vault = '' # Replace with actual vault

# vault = 'C:' # Replace with actual vault
# notes = [] # Replace with actual notes

html_data = [html_data_from_note(note, vault) for note in notes]
max_length = 1022

tokenized_html_data = [tokenize_html_data(html_locus, tokenizer, max_length, def_or_notat_from_html_tag, label2id) for html_locus in html_data]
token_id_data = [token_ids for token_ids, _ in tokenized_html_data]
ner_tag_data = [ner_tag_ids for _, ner_tag_ids in tokenized_html_data]
token_seqs = [token_seq for token_seq in token_ids for token_ids in token_id_data]
ner_tag_seqs = [ner_tag_seq for ner_tag_seq in ner_tag_ids for ner_tag_ids in ner_tag_data]

In [ ]:
#| notest
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
max_length = 1022
label2id = {
    "O": 0,
    "B-definition": 1,
    "I-definition": 2,
    "B-notation": 3,
    "I-notation": 4
} 
id2label = {value: key for key, value in label2id.items()}

In [ ]:
#| notest
note_names, token_seqs, ner_tag_seqs = [], [], []
for html_locus, (token_ids, ner_tag_ids) in zip(html_data, tokenized_html_data):
    note_names.extend([html_locus["Note name"]] * len(token_ids))
    token_seqs.extend(token_ids)
    ner_tag_seqs.extend(ner_tag_ids)

In [ ]:
#| notest
# ner_tags = ClassLabel(names=list(label2id))

# ds = Dataset.from_dict(
#         {"note_name": note_names,
#         "tokens": token_ids,
#         "ner_tags": ner_tag_ids},
#         features=Features(
#             {
#              "note_name": Value(dtype='string'),
#              "tokens": Sequence(Value(dtype='string')),
#              "ner_tags": Sequence(ner_tags)}
#         ))

# ds.save_to_disk(".")

# ds.load_from_disk(".")
    